# GIC Basics

Introduces the GIC module for computing geomagnetically induced currents
in power systems. The notebook walks through configuring a uniform E-field
storm, retrieving transformer GIC currents, visualizing the distribution,
and sweeping storm direction to identify the worst-case orientation.

Import the case and instantiate the `PowerWorld`.

```python
from esapp import PowerWorld
from esapp.components import *

pw = PowerWorld(case_path)
```

In [ ]:
# This cell is hidden in the documentation.
from esapp import PowerWorld
from esapp.components import *
import numpy as np
import matplotlib.pyplot as plt
import ast

with open('../data/case.txt', 'r') as f:
    case_path = ast.literal_eval(f.read().strip())

pw = PowerWorld(case_path)

In [ ]:
# Plotting functions (hidden from documentation)
import sys; sys.path.insert(0, "..")
from plot_helpers import plot_gic_distribution, plot_direction_sensitivity

## Calculate GIC Response

Compute geomagnetically induced currents for a uniform electric field. This calculates harmonic currents in transformers due to a 1.0 V/km electric field oriented at 90 degrees:

In [ ]:
pw.gic.storm(max_field=1.0, direction=90.0)

## Retrieve GIC Results

Extract GIC neutral currents from the transformers to identify which components experience the largest impacts:

In [ ]:
gics = pw[GICXFormer, ['BusNum3W', 'BusNum3W:1', 'GICXFNeutralAmps']]
gics.head()

In [ ]:
max_gic = gics['GICXFNeutralAmps'].abs().max()
print(f"Maximum |GIC|: {max_gic:.3f} Amps")

### GIC Distribution

Visualize the distribution of GIC magnitudes across all transformers.

In [ ]:
gic_abs = gics['GICXFNeutralAmps'].abs().sort_values(ascending=False)
plot_gic_distribution(gic_abs)

## Storm Direction Sensitivity

Sweep the E-field direction to find which orientation produces the worst-case GIC.

In [ ]:
directions = np.arange(0, 361, 10)
max_gics = []

for d in directions:
    pw.gic.storm(max_field=1.0, direction=d)
    gic_vals = pw[GICXFormer, 'GICXFNeutralAmps']['GICXFNeutralAmps']
    max_gics.append(gic_vals.abs().max())

max_gics = np.array(max_gics)

In [ ]:
plot_direction_sensitivity(directions, max_gics)